In [ ]:
import pandas as pd
import numpy as np
import functions_for_analyze as ffa
import single_output_parser as sop
import os
from types import SimpleNamespace
import matplotlib.pyplot as pp
from importlib import reload
from matplotlib.colors import TABLEAU_COLORS 
import _daily_reporters as dr
import json
from __visuals__ import matplot
from __visuals__ import _printing_and_display_functions as pad
import _interaction_functions as IF
import _Plot_from_Tables as PFT
import Make_Interaction_Tables as MIT
import pickle

%matplotlib widget

In [ ]:
style_sheet = "./__visuals__/f12xg_plot_style.mplstyle"
pp.style.use(style_sheet)

In [ ]:
def gen_fig_title(sysd, plot_type="Interaction energies"):
    return f"{sysd['mono1_name']}-{sysd['mono2_name']} {plot_type}"

def gen_fig_fname(sysd, ext='svg', plot_type="Interaction_energies", folder=None):
    fname= f"{sysd['mono1_name']}_{sysd['mono2_name']}_{plot_type}.{ext}"
    if folder is None:
        folder = '__Analysis_Outputs__/'
    fname = os.path.join(folder, fname)
    return fname

In [ ]:
def infer_cores(sysd):
    """Infer the 'cores' list by reading the dimer's bse_data_*.csv metacsv (per base.json)."""
    dimer_bse_datafile = f"{sysd['dimer_dir']}/{sysd['method_dir']}/{sysd['basename']}"
    _, dimer_df = ffa.MakeTables.read_metacsv(
        dimer_bse_datafile, kwargs2=sysd['dimer_kwargs2']
    )
    return sorted(set(dimer_df.index.get_level_values(0)))

def build_sysd(mono1_dir, mono2_dir, dimer_dir, method_dir, bases_family,
               base_json='queryFiles/base.json', mono1_name=None, mono2_name=None):
    """Build a sysd dict via Make_Interaction_Tables.build_sysd, with 'cores' inferred from disk."""
    base = MIT.load_base_config(base_json)
    args = SimpleNamespace(
        mono1_dir=mono1_dir, mono2_dir=mono2_dir,
        dimer_dir=dimer_dir, method_dir=method_dir,
        bases_family=bases_family, base_json=base_json,
        mono1_name=mono1_name, mono2_name=mono2_name,
    )
    sysd, outer_index = MIT.build_sysd(args, base)
    sysd['outer_index'] = outer_index
    sysd['cores'] = infer_cores(sysd)
    return sysd, outer_index

In [ ]:
def select_from_plot_data(plot_data, **match):
    """
    Select entries from a collect_interaction_data-style plot_data list
    ([xdata, ydata, labels], ...) whose labels match all of the given
    key=value pairs. Returns fresh (list) copies so the originals and the
    caller's added labels (e.g. a 'method' tag) don't interfere.
    """
    selected = []
    for x, y, labels in plot_data:
        if all(labels.get(k) == v for k, v in match.items()):
            selected.append([x, y, dict(labels)])
    return selected

In [ ]:
def get_available_gammas(sysd):
    """
    Discover the outer_index (e.g. 'gammas') values actually present on disk
    for this sysd, by reading one core's combined csv without slicing down
    to a single value. Excludes the 'Difference' pseudo-value added by
    combine_systems/add_sum_and_difference.
    """
    outer_index = sysd['outer_index']
    core = sysd['cores'][0]
    fname_sysd = dict(sysd, core=core)
    fname = IF.get_combined_outfile_name(fname_sysd)
    _, df = ffa.MakeTables.read_metacsv(
        fname, kwargs2=dict(header=[0, 1], index_col=[0, 1, 2])
    )

    values = []
    for v in df.index.get_level_values(outer_index).unique():
        try:
            values.append(float(v))
        except ValueError:
            continue
    return sorted(values)

# DFMP2 for Ne-Ar interactions

In [ ]:
method = 'dfmp2'
mono1_dir = 'ne'
mono2_dir = 'ar'
dimer_dir = 'ne_ar'

In [ ]:
reload(PFT)
reload(MIT)
sysd_core_valence, _ = build_sysd(mono1_dir, mono2_dir, dimer_dir, method, bases_family='core-valence')
dfs_core_valence = PFT.get_dfs(sysd_core_valence)

# ==== Now pVXZ basis
sysd_pV, _ = build_sysd(mono1_dir, mono2_dir, dimer_dir, method, bases_family='valence')
dfs_pV = PFT.get_dfs(sysd_pV)
#plot_core

## Interaction energies

### Filter and get plot data

In [ ]:
filter_specs=dict(
    cross_filters={
        "func_kwargs": [
            dict(totkey=("HF + ", "CorrE(CBS)")), 
            dict(totkey=("Reference Energy", "5"))
        ],
        "core": ["all electron", "frozen"],
    },
)

In [ ]:
reload(IF)
filter_specs['common_filter'] = {"basis": "aug-cc-pVXZ"}
filters = ffa.expand_filter_grid(**filter_specs)
plot_data = IF.collect_interaction_data(dfs_pV, filters)
filter_specs['common_filter'] = {"basis": "aug-cc-pwCVXZ"}
filters = ffa.expand_filter_grid(**filter_specs)
plot_data += IF.collect_interaction_data(dfs_core_valence, filters)


plot_data_dfmp2 = list(plot_data)

### Plot with Legend

In [ ]:
styles0 = {# common styles for all
    "basis" : {
        "aug-cc-pVXZ" : dict(color='tab:blue', label='pVXZ', linewidth=2),
        "aug-cc-pwCVXZ" : dict(color='tab:orange', label='pwCVXZ'),
    },
    "core" : {
        "frozen" : dict(marker='x', label='frozen'),
        "all electron"  : dict(marker='s', fillstyle='none', label='all e')
    }
}

styles = dict(styles0)
styles.update(
    {
        "totkey": {
            ("HF + ", "CorrE(CBS)") : dict(linestyle='-', label='HF5+CBS'),
            ("Reference Energy", "5") : dict(linestyle=':', label='HF5'),
        }
    }
)
                                         

In [ ]:
reload(PFT)
fig, ax, leg_ax = PFT.plot_with_external_style_legend(plot_data, styles, convert=10**(3))
ax.set_ylim(-0.5, 0.2)
ax.set_ylabel(rf"Energy (mH)")
ax.set_xlim(3,4.25)
ax.set_xlabel(r"Distance, $r$ $(\AA)$")
fig.suptitle(gen_fig_title(sysd_pV))

## Summary

* Frozen and all electron interaction energies are similar for pwCVXZ
* DFMP2 with pVXZ overestimates binding energy in the CBS limit for all electron
* But the frozen core approximation gives similar results to pwCVXZ case

### Take away: Use `DFMP2 aug-cc-pwCVXZ HF5 + CBS` as reference
  

In [ ]:
ref_plot_data_dfmp2 = []
for x,y,kwargs in plot_data_dfmp2:
    print(kwargs)
    if kwargs['basis'] == 'aug-cc-pwCVXZ' and kwargs['totkey'] == ('HF + ', 'CorrE(CBS)'):
        ref_plot_data_dfmp2.append([x, y, kwargs])

In [ ]:
for x,y,kwargs in ref_plot_data_dfmp2:
    print(kwargs)

# F12

In [ ]:
def assign_new_total(df, key1, key2, label):
    """Returns a fresh copy of df with `label` assigned as key2+key1.

    Always copies df first: some upstream dfs (e.g. dfs_xg's per
    core/gamma_set .xs() slices) are still flagged by pandas as
    possibly-a-view, which makes an in-place `df[label] = ...` raise
    SettingWithCopyWarning. Copying up front sidesteps that regardless of
    how the caller's df was produced.
    """
    df = df.copy()
    _, cbs_df = IF.get_HF_and_best_from_multimer_df(df, key1=key1, key2=key2)
    df[label] = cbs_df
    return df

In [ ]:
method = 'f12'
mono1_dir = 'ne'
mono2_dir = 'ar'
dimer_dir = 'ne_ar'

In [ ]:
sysd_core_valence, _ = build_sysd(mono1_dir, mono2_dir, dimer_dir, method, bases_family='core-valence')
sysd_pV, _ = build_sysd(mono1_dir, mono2_dir, dimer_dir, method, bases_family='valence')

gammas_f12 = get_available_gammas(sysd_pV)
print('Available F12 gammas:', gammas_f12)

dfs_core_valence_by_gamma = {g: PFT.get_dfs(sysd_core_valence, gamma=g) for g in gammas_f12}
dfs_pV_by_gamma = {g: PFT.get_dfs(sysd_pV, gamma=g) for g in gammas_f12}

### Assign new total energy

In [ ]:
keys_and_label = [('Correlation Energy', '4'), ('Reference Energy', '4'), ('HF4 +', 'CorrE(4)')]

for gamma in gammas_f12:
    for dfs in (dfs_core_valence_by_gamma[gamma], dfs_pV_by_gamma[gamma]):
        for core, values in dfs.items():
            values['df'] = assign_new_total(values['df'], *keys_and_label)

### Filter and get plot data

In [ ]:
filter_specs = dict(
    cross_filters={
        "func_kwargs": [
            dict(totkey=("HF + ", "CorrE(CBS)")),
            dict(totkey=keys_and_label[-1]),
        ],
        "core": ["frozen"],
    },
)

In [ ]:
plot_data_f12_by_gamma = {}
for gamma in gammas_f12:
    filter_specs['common_filter'] = {"basis": "aug-cc-pVXZ", "gammas": gamma}
    filters = ffa.expand_filter_grid(**filter_specs)
    plot_data = IF.collect_interaction_data(dfs_pV_by_gamma[gamma], filters)
    filter_specs['common_filter'] = {"basis": "aug-cc-pwCVXZ", "gammas": gamma}
    filters = ffa.expand_filter_grid(**filter_specs)
    plot_data += IF.collect_interaction_data(dfs_core_valence_by_gamma[gamma], filters)
    plot_data_f12_by_gamma[gamma] = plot_data

# XG

In [ ]:
method = 'xg'
mono1_dir = 'ne'
mono2_dir = 'ar'
dimer_dir = 'ne_ar'

In [ ]:
mono1_name = MIT.default_name_from_dir(mono1_dir)
mono2_name = MIT.default_name_from_dir(mono2_dir)
sysd_xg = dict(mono1_name=mono1_name, mono2_name=mono2_name)

# XG has no monomer bse_data yet, and no combined dimer_and_monomer csv (see TODO.md).
# Quick-and-dirty proxy: use the dimer's own row at distances=9999.0 as a stand-in
# for the monomer sum, instead of loading/summing separate monomer files.
reload(IF)
dimer_bse_datafile = f"{dimer_dir}/{method}/bse_data_valence.csv"
_, df_xg = ffa.MakeTables.read_metacsv(
    dimer_bse_datafile,
    kwargs2=dict(header=[0,1], index_col=[0,1,2])
)

# xg valence only goes up to QZ (no 5Z), so basis_size 4 is the best available HF reference
_, cbs = IF.get_HF_and_best_from_multimer_df(df_xg, key2=('Reference Energy', '4'))
df_xg['HF + ', 'CorrE(CBS)'] = cbs

cores = sorted(set(df_xg.index.get_level_values('core')))
gamma_sets_xg = sorted(set(df_xg.index.get_level_values('gamma_set')))
print('Available XG gamma_sets:', gamma_sets_xg)

dfs_xg_by_gamma_set = {
    gs: {
        core: {"meta": {}, "df": df_xg.xs((core, gs), level=('core', 'gamma_set'))}
        for core in cores
    }
    for gs in gamma_sets_xg
}

### Assign new total energy

In [ ]:
for gs in gamma_sets_xg:
    for core, values in dfs_xg_by_gamma_set[gs].items():
        values['df'] = assign_new_total(values['df'], *keys_and_label)

### Filter and get plot data

In [ ]:
plot_data_xg_by_gamma_set = {}
for gs in gamma_sets_xg:
    filter_specs['common_filter'] = {"basis": "aug-cc-pVXZ", "gamma_set": gs}
    filters = ffa.expand_filter_grid(**filter_specs)
    plot_data_xg_by_gamma_set[gs] = IF.collect_interaction_data(
        dfs_xg_by_gamma_set[gs], filters, func=IF.get_inter_from_distance_proxy
    )

# Comparison: DFMP2 vs F12 vs XG (HF+CBS)

In [ ]:
chosen_gamma = {
    "F12": 1.0,
    "XG": "1.00_1.00_1.00",
}

compare_totkey_dict = {
    "DFMP2": ("HF + ", "CorrE(CBS)"),
    "F12": keys_and_label[-1],
    "XG": keys_and_label[-1],
}

compare_basis_dict = {
    "DFMP2": "aug-cc-pwCVXZ",
    "F12": "aug-cc-pVXZ",
    "XG": "aug-cc-pVXZ",
}

plot_data_dict = {
    "DFMP2": plot_data_dfmp2,
    "F12": plot_data_f12_by_gamma[chosen_gamma["F12"]],
    "XG": plot_data_xg_by_gamma_set[chosen_gamma["XG"]],
}

plot_data_compare = []
for method_label, pdata in plot_data_dict.items():
    subset = select_from_plot_data(
        pdata,
        totkey=compare_totkey_dict[method_label],
        basis=compare_basis_dict[method_label],
        core="frozen",
    )
    for _, _, labels in subset:
        labels["method"] = method_label
    plot_data_compare += subset

In [ ]:
styles_compare = {
    "method": {
        "DFMP2": dict(color='tab:blue', linestyle='--', marker='s', fillstyle="none", label='DFMP2'),
        "F12": dict(color='tab:green', marker='+', label=f'F12 (gamma={chosen_gamma["F12"]})'),
        "XG": dict(color='tab:red', marker='x', label=f'XG (gamma_set={chosen_gamma["XG"]})'),
    },
}

### Plot with Legend

In [ ]:
reload(PFT)
fig, ax, leg_ax = PFT.plot_with_external_style_legend(plot_data_compare, styles_compare, convert=10**(3))
ax.set_ylabel(rf"Energy (mH)")
ax.set_xlabel(r"Distance, $r$ $(\AA)$")
ax.set_ylim(-0.25, 0.05)
ax.set_xlim(3.1,4.)
fig.suptitle(gen_fig_title(sysd_xg, plot_type="Interaction energies: DFMP2(HF+CBS) vs F12 vs XG (frozen core)"))
fig.set_size_inches(12,8)